```
╔══════════════════════════════════════════════════════════════════════════════╗
║                                                                              ║
║   SISTEMA DE FORECAST PREDICTIVO DE FALLAS — V4 UNIFICADO                   ║
║   Planta de Motores Silao — Volkswagen de México                            ║
║                                                                              ║
║   FLUJO DE EJECUCIÓN (UNA SOLA PASADA):                                      ║
║     1. Coloca el archivo del mes nuevo (NUEVO_DATA_FILE)                     ║
║     2. Ejecuta TODO el notebook (Run All)                                    ║
║     3. El sistema automáticamente:                                           ║
║        a) Valida y combina datos históricos + nuevos                         ║
║        b) Re-entrena con warm-start (~15 seg) usando features V4             ║
║        c) Genera forecast recursivo a 3 meses                                ║
║        d) Produce las 9 visualizaciones ejecutivas idénticas                 ║
║        e) Exporta los 3 CSVs para Power BI                                   ║
║        f) Versiona el modelo con rollback disponible                         ║
║                                                                              ║
║   FEATURES V4 (vs V3):                                                       ║
║     + Engine code letter  (chi²=184, reemplaza EA-Number)                    ║
║     + age_days / engine_age_days desde fechas (F≈274)                        ║
║     + KM_per_MIS: intensidad de uso (F=144)                                  ║
║     + DAMAGE CODE, REPAIR, ISO country, Displacement                         ║
║     + component_repeat_rate: target encoding con smoothing bayesiano         ║
║                                                                              ║
║   DEPENDENCIAS:                                                               ║
║     conda install -c conda-forge xgboost jupyterlab seaborn                  ║
║     pip install pandas numpy scikit-learn openpyxl joblib matplotlib         ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
```

## Celda 1 — Imports y configuración global

In [ ]:
import pandas as pd
import numpy as np
import warnings
import logging
import json
import shutil
import time
from datetime import datetime
from pathlib import Path
import joblib

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch
import seaborn as sns

from xgboost import XGBRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_percentage_error

warnings.filterwarnings('ignore')
logging.getLogger('xgboost').setLevel(logging.ERROR)

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN — ajusta estos valores antes de ejecutar
# ══════════════════════════════════════════════════════════════════════════════

# Archivo con TODOS los datos históricos acumulados
DATA_FILE         = 'Datos_Power_BI_-_Silao.xlsx'

# Archivo con los datos del mes nuevo a incorporar
# Si no existe, el notebook usa solo DATA_FILE (primer entrenamiento)
NUEVO_DATA_FILE   = 'Datos_Power_BI_-_Silao_NUEVO.xlsx'

MODEL_DIR         = Path('modelo_v4')
OUTPUT_DIR        = Path('outputs_v4')
HISTORY_DIR       = Path('historial_modelos')
N_MESES_FC        = 3       # Horizonte de forecast
TOP_N             = 50      # Combinaciones en el ranking
N_ARBOLES_EXTRA   = 100     # Árboles extra en warm-start
PESO_RECIENTES    = 2.0     # Peso de los últimos 3 meses al re-entrenar

for d in [MODEL_DIR, OUTPUT_DIR, HISTORY_DIR]:
    d.mkdir(exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════════
# Paleta corporativa Silao — VW Quality
# ══════════════════════════════════════════════════════════════════════════════
COLOR_NAVY   = '#0A2540'
COLOR_TEAL   = '#1E5F74'
COLOR_GOLD   = '#D4A74A'
COLOR_GREEN  = '#16A085'
COLOR_RED    = '#C0392B'
COLOR_ORANGE = '#E67E22'
COLOR_YELLOW = '#F39C12'
COLOR_LIGHT  = '#F7F5F0'
COLOR_GRAY_D = '#2D3E50'
COLOR_GRAY_M = '#5A6978'
COLOR_GRAY_L = '#D8DEE4'

MAPA_NIVELES = {
    '🔴 CRÍTICO': COLOR_RED,
    '🟠 ALTO':    COLOR_ORANGE,
    '🟡 MEDIO':   COLOR_YELLOW,
    '🟢 BAJO':    COLOR_GREEN,
}
MAPA_NIVELES_TEXTO = {
    '🔴 CRÍTICO': 'CRÍTICO', '🟠 ALTO': 'ALTO',
    '🟡 MEDIO': 'MEDIO',    '🟢 BAJO': 'BAJO',
}

plt.rcParams.update({
    'font.family':'DejaVu Sans','font.size':11,
    'axes.titlesize':14,'axes.titleweight':'bold','axes.titlepad':14,
    'axes.labelsize':11,'axes.spines.top':False,'axes.spines.right':False,
    'axes.edgecolor':COLOR_GRAY_M,'axes.linewidth':0.8,
    'axes.grid':True,'grid.color':COLOR_GRAY_L,'grid.linewidth':0.5,'grid.alpha':0.6,
    'xtick.color':COLOR_GRAY_D,'ytick.color':COLOR_GRAY_D,
    'legend.frameon':False,'legend.fontsize':10,
    'figure.facecolor':'white','axes.facecolor':'white',
    'savefig.facecolor':'white','savefig.dpi':150,'savefig.bbox':'tight',
})

def fmt_currency_k(x, _):
    if abs(x)>=1e6: return f'${x/1e6:.1f}M'
    if abs(x)>=1e3: return f'${x/1e3:.0f}k'
    return f'${x:.0f}'

def styled_title(ax, title, subtitle=None):
    ax.set_title('')
    y0 = 1.10 if subtitle else 1.04
    ax.text(0, y0, title, transform=ax.transAxes, fontsize=14, fontweight='bold', color=COLOR_NAVY)
    if subtitle:
        ax.text(0, 1.04, subtitle, transform=ax.transAxes, fontsize=10, fontstyle='italic', color=COLOR_GRAY_M)

print('✅ Configuración lista')
print(f'   Datos base:   {DATA_FILE}')
print(f'   Datos nuevos: {NUEVO_DATA_FILE}  (existe: {Path(NUEVO_DATA_FILE).exists()})')

## Celda 2 — Funciones de carga, features V4 y agregación mensual

In [ ]:
MESES_ES = {
    'enero':1,'febrero':2,'marzo':3,'abril':4,'mayo':5,'junio':6,
    'julio':7,'agosto':8,'septiembre':9,'octubre':10,'noviembre':11,'diciembre':12
}

def parse_fecha(s):
    if pd.isna(s): return pd.NaT
    try:
        p = str(s).lower().split(', ',1)[-1].split(' de ')
        return pd.Timestamp(int(p[2]), MESES_ES[p[1]], int(p[0]))
    except: return pd.NaT

def cargar_y_enriquecer(df_input):
    """
    Recibe un DataFrame ya cargado (o ruta string) y aplica todo el
    preprocesamiento V4: fechas, features derivadas, costos limpios.
    """
    if isinstance(df_input, (str, Path)):
        df = pd.read_excel(df_input)
    else:
        df = df_input.copy()

    df['fecha']     = df['Repair date'].apply(parse_fecha)
    df['prod_dt']   = df['Production Date'].apply(parse_fecha)
    df['engine_dt'] = df['Engine Date'].apply(parse_fecha)

    df = df.dropna(subset=['fecha','Basic no. Description','TMA description',
                            'MIS','Engine code letter'])

    max_mes = df['fecha'].dt.to_period('M').max()
    df = df[df['fecha'].dt.to_period('M') < max_mes]

    # Features derivadas de fecha (F≈274)
    df['age_days']        = (df['fecha'] - df['prod_dt']).dt.days.clip(lower=0)
    df['engine_age_days'] = (df['fecha'] - df['engine_dt']).dt.days.clip(lower=0)
    df['KM_per_MIS']      = df['KM'] / (df['MIS'] + 1)

    df['MIS_Cluster'] = df['MIS'].apply(
        lambda m: '0-3 MIS' if m<=3 else ('4-12 MIS' if m<=12 else '13-24 MIS'))

    df['costo_material'] = df['Material cost'].clip(lower=0)
    df['costo_labour']   = df['Labour cost'].clip(lower=0)
    df['costo_total']    = df['costo_material'] + df['costo_labour']

    df['DAMAGE CODE']         = df['DAMAGE CODE'].fillna('UNKNOWN')
    df['REPAIR']               = df['REPAIR'].fillna('ers.')
    df['ISO country code']    = df['ISO country code'].fillna('USA')
    df['Displacement litres'] = df['Displacement litres'].fillna(
        df['Displacement litres'].median())

    df['mes'] = df['fecha'].dt.to_period('M').dt.start_time
    return df

def calcular_component_repeat_rate(df):
    """Target encoding con smoothing bayesiano."""
    global_mean = (df['REPEAT REPAIR FLG'] == 'J').mean()
    k = 10
    stats = df.groupby('Basic no. Description').agg(
        n=('REPEAT REPAIR FLG','count'),
        s=('REPEAT REPAIR FLG', lambda x: (x=='J').sum())
    )
    stats['rate'] = (stats['s'] + k*global_mean) / (stats['n'] + k)
    return stats['rate'].to_dict(), global_mean

def agregar_mensual(df):
    monthly = df.groupby(
        ['mes','Basic no. Description','TMA description',
         'Engine code letter','MIS_Cluster']
    ).agg(
        fallas          = ('costo_total','count'),
        costo_total     = ('costo_total','sum'),
        material_total  = ('costo_material','sum'),
        labour_total    = ('costo_labour','sum'),
        avg_km          = ('KM','mean'),
        avg_mis         = ('MIS','mean'),
        avg_age_days    = ('age_days','mean'),
        avg_engine_age  = ('engine_age_days','mean'),
        avg_km_per_mis  = ('KM_per_MIS','mean'),
        avg_displacement= ('Displacement litres','mean'),
        avg_repeat_rate = ('component_repeat_rate','mean'),
        mode_damage     = ('DAMAGE CODE',     lambda x: x.mode()[0] if len(x)>0 else 'UNKNOWN'),
        mode_repair     = ('REPAIR',           lambda x: x.mode()[0] if len(x)>0 else 'ers.'),
        mode_country    = ('ISO country code', lambda x: x.mode()[0] if len(x)>0 else 'USA'),
    ).reset_index()
    monthly.columns = [
        'mes','codigo','tma','engine_code','mis_cluster',
        'fallas','costo_total','material_total','labour_total',
        'avg_km','avg_mis','avg_age_days','avg_engine_age',
        'avg_km_per_mis','avg_displacement','avg_repeat_rate',
        'mode_damage','mode_repair','mode_country'
    ]
    monthly['mes'] = pd.to_datetime(monthly['mes'])
    return monthly.sort_values(['codigo','tma','engine_code','mis_cluster','mes']).reset_index(drop=True)

def construir_costos_referencia(df, ventana_meses=3):
    df_r = df.copy()
    df_r['ym'] = df_r['fecha'].dt.to_period('M')
    cutoff = df_r['ym'].max() - ventana_meses + 1
    rec = df_r[df_r['ym'] >= cutoff]
    ref = rec.groupby(['Basic no. Description','TMA description']).agg(
        costo_unit_med = ('costo_total','median'),
        costo_unit_avg = ('costo_total','mean'),
        costo_unit_p90 = ('costo_total', lambda x: x.quantile(0.9)),
        material_unit  = ('costo_material','mean'),
        labour_unit    = ('costo_labour','mean'),
        n_obs          = ('costo_total','count'),
    ).reset_index().rename(columns={
        'Basic no. Description':'codigo','TMA description':'tma'
    })
    fallback = df_r.groupby('Basic no. Description')['costo_total'].median().to_dict()
    return ref, fallback

def winsorize(s, lo=0.02, hi=0.98):
    return s.clip(s.quantile(lo), s.quantile(hi))

FEATS = [
    'cod_enc','tma_enc','mis_enc','engine_enc',
    'lag1','lag2','lag3','lag4','lag5','lag6',
    'roll3','roll3std','roll6','roll6std',
    'ema3','ema6',
    'mom',
    't_norm','msin','mcos','g_idx_lag1','g_idx_chg','log_mean_hist',
    'yoy_ratio',
    'damage_enc','repair_enc','country_enc',
    'avg_age_days','avg_engine_age','avg_km_per_mis',
    'avg_displacement','avg_repeat_rate',
]

def construir_features(monthly_df):
    filas = []
    for _, grp in monthly_df.groupby(['cod_enc','tma_enc','engine_enc','mis_enc']):
        g = grp.sort_values('mes').reset_index(drop=True)
        if len(g) < 8: continue
        g['fallas_w'] = winsorize(g['fallas'])
        g['scale'] = g['fallas_w'].shift(1).rolling(6,min_periods=2).mean().bfill().fillna(1).clip(lower=0.1)
        g['y_scaled'] = g['fallas_w'] / g['scale']
        for lag in [1, 2, 3, 4, 5, 6]:
            g[f'lag{lag}'] = g['fallas_w'].shift(lag) / g['scale'].shift(lag).clip(lower=0.1)
        g['roll3']    = g['fallas_w'].shift(1).rolling(3).mean() / g['scale'].shift(1).clip(lower=0.1)
        g['roll3std'] = g['fallas_w'].shift(1).rolling(3).std().fillna(0) / g['scale'].shift(1).clip(lower=0.1)
        g['roll6']    = g['fallas_w'].shift(1).rolling(6).mean() / g['scale'].shift(1).clip(lower=0.1)
        g['roll6std'] = g['fallas_w'].shift(1).rolling(6).std().fillna(0) / g['scale'].shift(1).clip(lower=0.1)
        fw_s1 = g['fallas_w'].shift(1)
        g['ema3'] = fw_s1.ewm(span=3, min_periods=1, adjust=False).mean() / g['scale'].shift(1).clip(lower=0.1)
        g['ema6'] = fw_s1.ewm(span=6, min_periods=1, adjust=False).mean() / g['scale'].shift(1).clip(lower=0.1)
        g['mom']  = g['lag1'] - g['lag2']
        g['yoy_ratio'] = (
            g['fallas_w'].shift(1) / g['fallas_w'].shift(12).clip(lower=0.1)
        ).clip(0.05, 20).fillna(1.0)
        t_g           = (g['mes'].dt.year-2024)*12 + g['mes'].dt.month
        g['t_norm']   = t_g / 27.0
        g['msin']     = np.sin(2*np.pi*g['mes'].dt.month/12)
        g['mcos']     = np.cos(2*np.pi*g['mes'].dt.month/12)
        g['g_idx_lag1']    = g['global_idx'].shift(1)
        g['g_idx_chg']     = g['global_idx'] - g['global_idx'].shift(1)
        g['log_mean_hist'] = np.log1p(g['fallas_w'].expanding(min_periods=3).mean())
        for feat in ['avg_age_days','avg_engine_age','avg_km_per_mis',
                     'avg_displacement','avg_repeat_rate',
                     'damage_enc','repair_enc','country_enc']:
            if feat in g.columns:
                g[feat] = g[feat].shift(1).bfill()
        filas.append(g)
    return pd.concat(filas, ignore_index=True).dropna(subset=FEATS+['y_scaled'])

print('✅ Funciones de preprocesamiento y features V4 definidas')
print(f'   Total features: {len(FEATS)}  (+7 vs V4)')

## Celda 3 — Carga inteligente: detecta si hay datos nuevos y combina

In [ ]:
hay_datos_nuevos = Path(NUEVO_DATA_FILE).exists()
hay_modelo_previo = (MODEL_DIR / 'model_central.pkl').exists()

print('══════════════════════════════════════════════════════════')
print('  DETECCIÓN DE MODO DE EJECUCIÓN')
print('══════════════════════════════════════════════════════════')
print(f'  Datos nuevos disponibles: {hay_datos_nuevos}')
print(f'  Modelo previo disponible: {hay_modelo_previo}')

if hay_datos_nuevos:
    modo = 'INCREMENTAL' if hay_modelo_previo else 'INICIAL_CON_NUEVO'
else:
    modo = 'INICIAL'

print(f'\n  → MODO: {modo}')
if modo == 'INCREMENTAL':
    print('  → Se combinará el histórico con los datos nuevos')
    print('  → Warm-start sobre el modelo existente (~15 seg)')
elif modo == 'INICIAL_CON_NUEVO':
    print('  → Primer entrenamiento usando datos base + nuevos combinados')
else:
    print('  → Primer entrenamiento usando solo DATA_FILE')

# ── Combinar datos si corresponde ─────────────────────────────────────────
if hay_datos_nuevos:
    print('\n🔍 Validando y combinando datos...')
    df_hist  = pd.read_excel(DATA_FILE)
    df_nuevo = pd.read_excel(NUEVO_DATA_FILE)

    # Validar schema
    cols_h = set(df_hist.columns)
    cols_n = set(df_nuevo.columns)
    if cols_h != cols_n:
        raise ValueError(f'Schema incompatible. Faltantes: {cols_h-cols_n}. Extras: {cols_n-cols_h}')

    # Detectar solape temporal
    df_hist['_f']  = df_hist['Repair date'].apply(parse_fecha)
    df_nuevo['_f'] = df_nuevo['Repair date'].apply(parse_fecha)
    meses_nuevo = set(df_nuevo['_f'].dt.to_period('M').dropna().unique())
    meses_hist  = set(df_hist['_f'].dt.to_period('M').dropna().unique())
    solape      = meses_hist & meses_nuevo
    genuinos    = meses_nuevo - meses_hist

    if solape:
        print(f'  ⚠  {len(solape)} meses solapados → reemplazados por versión nueva')
    print(f'  ✓  {len(genuinos)} meses genuinamente nuevos: {sorted([str(m) for m in genuinos])}')

    # Combinar (el nuevo gana en caso de solape)
    mask = ~df_hist['_f'].dt.to_period('M').isin(meses_nuevo)
    df_combinado = pd.concat([df_hist[mask], df_nuevo], ignore_index=True
                              ).drop(columns=['_f'], errors='ignore')
    df_hist.drop(columns=['_f'], inplace=True, errors='ignore')

    print(f'  ✓  Total combinado: {len(df_combinado):,} registros')

    # Guardar combinado como nuevo histórico
    df_combinado.to_excel('Datos_Power_BI_-_Silao_combinado.xlsx', index=False)
    archivo_a_procesar = df_combinado
else:
    archivo_a_procesar = DATA_FILE

# ── Procesar con features V4 ─────────────────────────────────────────────
print('\n📐 Aplicando preprocesamiento V4...')
df_raw = cargar_y_enriquecer(archivo_a_procesar)

# Target encoding de componente
crr_map, global_rr = calcular_component_repeat_rate(df_raw)
df_raw['component_repeat_rate'] = df_raw['Basic no. Description'].map(crr_map).fillna(global_rr)

n_meses = df_raw['mes'].dt.to_period('M').nunique()
print(f'\n📂 Datos listos:')
print(f'   {len(df_raw):,} registros | {df_raw["fecha"].min().date()} → {df_raw["fecha"].max().date()}')
print(f'   {df_raw["Basic no. Description"].nunique()} componentes | {n_meses} meses | {df_raw["Engine code letter"].nunique()} engine codes')
print(f'   💰 Costo material: ${df_raw["costo_material"].sum():,.0f}')
print(f'   💰 Costo labour:   ${df_raw["costo_labour"].sum():,.0f}')

## Celda 4 — Encoders, agregación mensual y feature engineering

In [ ]:
monthly = agregar_mensual(df_raw)
costos_ref, costos_fallback = construir_costos_referencia(df_raw, ventana_meses=3)

# Encoders
le_cod     = LabelEncoder().fit(monthly['codigo'])
le_tma     = LabelEncoder().fit(monthly['tma'])
le_mis     = LabelEncoder().fit(monthly['mis_cluster'])
le_engine  = LabelEncoder().fit(monthly['engine_code'])
le_damage  = LabelEncoder().fit(monthly['mode_damage'])
le_repair  = LabelEncoder().fit(monthly['mode_repair'])
le_country = LabelEncoder().fit(monthly['mode_country'])

monthly['cod_enc']     = le_cod.transform(monthly['codigo'])
monthly['tma_enc']     = le_tma.transform(monthly['tma'])
monthly['mis_enc']     = le_mis.transform(monthly['mis_cluster'])
monthly['engine_enc']  = le_engine.transform(monthly['engine_code'])
monthly['damage_enc']  = le_damage.transform(monthly['mode_damage'])
monthly['repair_enc']  = le_repair.transform(monthly['mode_repair'])
monthly['country_enc'] = le_country.transform(monthly['mode_country'])

gt = monthly.groupby('mes')['fallas'].sum().reset_index()
gt['global_idx'] = gt['fallas'] / gt['fallas'].max()
monthly = monthly.merge(gt[['mes','global_idx']], on='mes', how='left')

feat_df = construir_features(monthly)

print(f'📊 Series mensuales: {monthly.shape}')
print(f'✅ Features V4 construidos: {feat_df.shape[0]:,} obs × {len(FEATS)} features')

## Celda 5 — Entrenamiento (inicial completo o warm-start incremental)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Celda 4b — Optuna HPO + XGB_PARAMS (ejecutar ANTES del entrenamiento)
# Busca hiperparámetros óptimos con TimeSeriesSplit. Si optuna no está
# instalado, usa los parámetros base definidos aquí como fallback.
# ══════════════════════════════════════════════════════════════════════════

# ── Parámetros base (fallback si Optuna no está disponible) ──────────────
XGB_PARAMS = dict(
    n_estimators=350, learning_rate=0.006600300898352391, max_depth=8,
    subsample=0.9765915797175485, colsample_bytree=0.9993274261356829,
    colsample_bylevel=0.9137806695369647,
    min_child_weight=5, reg_alpha=0.0010216951284402825,
    reg_lambda=0.1638592697378927, gamma=0.7688963243851896,
    random_state=42, verbosity=0,
)

try:
    import optuna
    from sklearn.model_selection import TimeSeriesSplit
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    _X_opt  = feat_df[FEATS].values
    _y_opt  = feat_df['y_scaled'].values
    _sc_opt = feat_df['scale'].values
    _ac_opt = feat_df['fallas'].values
    _tscv   = TimeSeriesSplit(n_splits=5)

    def _objective(trial):
        params = dict(
            n_estimators     = trial.suggest_int('n_estimators', 200, 700),
            learning_rate    = trial.suggest_float('learning_rate', 0.003, 0.05, log=True),
            max_depth        = trial.suggest_int('max_depth', 4, 10),
            subsample        = trial.suggest_float('subsample', 0.7, 1.0),
            colsample_bytree = trial.suggest_float('colsample_bytree', 0.6, 1.0),
            colsample_bylevel= 0.9137806695369647,
            min_child_weight = trial.suggest_int('min_child_weight', 1, 20),
            reg_alpha        = trial.suggest_float('reg_alpha', 1e-5, 1.0, log=True),
            reg_lambda       = trial.suggest_float('reg_lambda', 0.01, 2.0, log=True),
            gamma            = trial.suggest_float('gamma', 0.0, 2.0),
            random_state=42, verbosity=0,
        )
        search_p = {**params, 'n_estimators': min(200, params['n_estimators'])}
        fold_mapes = []
        for tr_idx, val_idx in _tscv.split(_X_opt):
            m_t = XGBRegressor(**search_p, objective='reg:squarederror')
            m_t.fit(_X_opt[tr_idx], _y_opt[tr_idx])
            pred_v = np.maximum(0, m_t.predict(_X_opt[val_idx])) * _sc_opt[val_idx]
            act_v  = _ac_opt[val_idx]
            mask_v2 = act_v >= 20
            if mask_v2.sum() > 0:
                fold_mapes.append(
                    mean_absolute_percentage_error(act_v[mask_v2], pred_v[mask_v2]) * 100
                )
        return np.mean(fold_mapes) if fold_mapes else 100.0

    print('🔍 Iniciando Optuna HPO (60 trials, timeout 120s)...')
    t0_hpo = time.time()
    study = optuna.create_study(direction='minimize',
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(_objective, n_trials=60, timeout=120, show_progress_bar=False)
    print(f'✅ HPO completado en {time.time()-t0_hpo:.0f}s | '
          f'{len(study.trials)} trials | Mejor CV MAPE: {study.best_value:.2f}%')

    best = study.best_params
    XGB_PARAMS = dict(
        n_estimators     = best['n_estimators'],
        learning_rate    = best['learning_rate'],
        max_depth        = best['max_depth'],
        subsample        = best['subsample'],
        colsample_bytree = best['colsample_bytree'],
        colsample_bylevel= 0.9137806695369647,
        min_child_weight = best['min_child_weight'],
        reg_alpha        = best['reg_alpha'],
        reg_lambda       = best['reg_lambda'],
        gamma            = best['gamma'],
        random_state=42, verbosity=0,
    )
    print(f'   XGB_PARAMS actualizado con resultados Optuna')

except ImportError:
    print('⚠  Optuna no instalado — pip install optuna — usando params predefinidos')
except Exception as e:
    print(f'⚠  HPO falló ({e}) — usando XGB_PARAMS predefinido')


In [ ]:
X = feat_df[FEATS]
y = feat_df['y_scaled']

# Pesos: decaimiento exponencial (suave) — mayor peso al mes más reciente
DECAY_HALF_LIFE = 6  # el peso se reduce a la mitad cada 6 meses
max_mes      = feat_df['mes'].max()
meses_atras  = ((max_mes - feat_df['mes']) / pd.Timedelta(days=30.44)).round().astype(float)
decay_rate   = np.log(2) / DECAY_HALF_LIFE
raw_w        = np.exp(-decay_rate * meses_atras)
w_min, w_max = raw_w.min(), raw_w.max()
sample_weights = 1.0 + (PESO_RECIENTES - 1.0) * (raw_w - w_min) / (w_max - w_min + 1e-9)

t0 = time.time()

if modo == 'INCREMENTAL':
    print('🔄 MODO INCREMENTAL — warm-start sobre modelo previo...')
    m_c_old = joblib.load(MODEL_DIR / 'model_central.pkl')
    m_l_old = joblib.load(MODEL_DIR / 'model_lower.pkl')
    m_u_old = joblib.load(MODEL_DIR / 'model_upper.pkl')
    enc_old = joblib.load(MODEL_DIR / 'encoders.pkl')

    # Verificar compatibilidad de encoders (nuevos codes/tmas?)
    cods_nuevos = set(monthly['codigo']) - set(enc_old.get('le_cod', le_cod).classes_)
    warm_ok = len(cods_nuevos) == 0

    if warm_ok:
        n_total = m_c_old.n_estimators + N_ARBOLES_EXTRA
        print(f'  ✓ Warm-start válido — {m_c_old.n_estimators} árboles base + {N_ARBOLES_EXTRA} nuevos')
        m_central = XGBRegressor(**{**XGB_PARAMS,'n_estimators':n_total,'objective':'reg:squarederror'})
        m_lower   = XGBRegressor(**{**XGB_PARAMS,'n_estimators':n_total,'objective':'reg:quantileerror','quantile_alpha':0.055})
        m_upper   = XGBRegressor(**{**XGB_PARAMS,'n_estimators':n_total,'objective':'reg:quantileerror','quantile_alpha':0.945})
        m_central.fit(X, y, sample_weight=sample_weights, xgb_model=m_c_old.get_booster())
        m_lower.fit(X, y,   sample_weight=sample_weights, xgb_model=m_l_old.get_booster())
        m_upper.fit(X, y,   sample_weight=sample_weights, xgb_model=m_u_old.get_booster())
    else:
        print(f'  ⚠ {len(cods_nuevos)} códigos nuevos → re-entrenamiento completo')
        m_central = XGBRegressor(**XGB_PARAMS, objective='reg:squarederror')
        m_lower   = XGBRegressor(**XGB_PARAMS, objective='reg:quantileerror', quantile_alpha=0.055)
        m_upper   = XGBRegressor(**XGB_PARAMS, objective='reg:quantileerror', quantile_alpha=0.945)
        m_central.fit(X, y, sample_weight=sample_weights)
        m_lower.fit(X, y,   sample_weight=sample_weights)
        m_upper.fit(X, y,   sample_weight=sample_weights)
else:
    print('🏋️  MODO INICIAL — entrenamiento completo desde cero...')
    m_central = XGBRegressor(**XGB_PARAMS, objective='reg:squarederror')
    m_lower   = XGBRegressor(**XGB_PARAMS, objective='reg:quantileerror', quantile_alpha=0.055)
    m_upper   = XGBRegressor(**XGB_PARAMS, objective='reg:quantileerror', quantile_alpha=0.945)
    m_central.fit(X, y)
    m_lower.fit(X, y)
    m_upper.fit(X, y)

t1 = time.time()

# Validación holdout (últimos 2 meses)
cutoff_val = feat_df['mes'].max() - pd.DateOffset(months=2)
te    = feat_df[feat_df['mes'] > cutoff_val]
pred  = np.maximum(0, m_central.predict(te[FEATS])) * te['scale'].values
actual = te['fallas'].values
mask_v = actual >= 20
mape   = mean_absolute_percentage_error(actual[mask_v], pred[mask_v]) * 100
accuracy = 100 - mape

print(f'\n✅ Modelo listo en {t1-t0:.1f}s')
print(f'   Accuracy holdout (fallas≥20): {accuracy:.1f}%')
print(f'   Series evaluadas: {mask_v.sum()}')

# ── TimeSeriesSplit CV (informativo, no reemplaza holdout) ────────────────
try:
    from sklearn.model_selection import TimeSeriesSplit
    _tscv_eval = TimeSeriesSplit(n_splits=5)
    _X_cv  = feat_df[FEATS].values
    _y_cv  = feat_df['y_scaled'].values
    _sc_cv = feat_df['scale'].values
    _ac_cv = feat_df['fallas'].values
    cv_mapes = []
    for fold, (tr_i, val_i) in enumerate(_tscv_eval.split(_X_cv)):
        m_cv = XGBRegressor(**XGB_PARAMS, objective='reg:squarederror')
        m_cv.fit(_X_cv[tr_i], _y_cv[tr_i], sample_weight=sample_weights[tr_i])
        pv = np.maximum(0, m_cv.predict(_X_cv[val_i])) * _sc_cv[val_i]
        av = _ac_cv[val_i]
        mk = av >= 20
        if mk.sum() > 0:
            fm = mean_absolute_percentage_error(av[mk], pv[mk]) * 100
            cv_mapes.append(fm)
            print(f'   CV fold {fold+1}/5 — MAPE: {fm:.1f}%  ({mk.sum()} series)')
    if cv_mapes:
        print(f'\n   CV MAPE (5-fold): {np.mean(cv_mapes):.1f}% ± {np.std(cv_mapes):.1f}%')
        print(f'   CV Accuracy:      {100-np.mean(cv_mapes):.1f}%')
except Exception as e:
    print(f'   (CV omitido: {e})')

## Celda 6 — Forecast recursivo a 3 meses

In [ ]:
def _ema(values, span):
    """EMA equivalente a pandas ewm(span, adjust=False)."""
    alpha = 2.0 / (span + 1.0)
    result = float(values[0])
    for v in values[1:]:
        result = alpha * float(v) + (1.0 - alpha) * result
    return result

def forecast_recursivo(monthly_df, mc, ml, mu, n_meses=3):
    historia = monthly_df.copy()
    historia['fallas_w'] = historia['fallas'].copy()
    forecasts = []
    print(f'🔮 Generando forecast recursivo a {n_meses} meses...')

    for paso in range(1, n_meses+1):
        next_month = historia['mes'].max() + pd.DateOffset(months=1)
        filas_pred = []

        for (c,t,e,m), g in historia.groupby(['cod_enc','tma_enc','engine_enc','mis_enc']):
            g = g.sort_values('mes').reset_index(drop=True)
            if len(g) < 4: continue
            ult   = g['fallas_w'].tail(13).values
            scale = max(0.1, np.mean(ult[-6:]) if len(ult)>=6 else np.mean(ult))
            def lm(col, n=3):
                return float(np.nanmean(g[col].tail(n).values)) if col in g.columns else 0.0
            tg = (next_month.year-2024)*12 + next_month.month
            row = {
                'cod_enc':c,'tma_enc':t,'engine_enc':e,'mis_enc':m,
                'mes':next_month,'scale':scale,
                'lag1':    ult[-1]/scale,
                'lag2':    ult[-2]/scale  if len(ult)>=2  else ult[-1]/scale,
                'lag3':    ult[-3]/scale  if len(ult)>=3  else ult[-1]/scale,
                'lag4':    ult[-4]/scale  if len(ult)>=4  else ult[-1]/scale,
                'lag5':    ult[-5]/scale  if len(ult)>=5  else ult[-1]/scale,
                'lag6':    ult[-6]/scale  if len(ult)>=6  else ult[-1]/scale,
                'roll3':   np.mean(ult[-3:])/scale,
                'roll3std':np.std(ult[-3:])/scale   if len(ult)>=3 else 0,
                'roll6':   np.mean(ult[-6:])/scale  if len(ult)>=6 else np.mean(ult)/scale,
                'roll6std':np.std(ult[-6:])/scale   if len(ult)>=6 else 0,
                'ema3':    _ema(ult, span=3)/scale,
                'ema6':    _ema(ult, span=6)/scale,
                't_norm':  tg/27.0,
                'msin':    np.sin(2*np.pi*next_month.month/12),
                'mcos':    np.cos(2*np.pi*next_month.month/12),
                'g_idx_lag1':    g['global_idx'].iloc[-1],
                'g_idx_chg':     0.02,
                'log_mean_hist': np.log1p(np.mean(ult)),
                'yoy_ratio': float(np.clip(
                    ult[-1] / max(ult[-13], 0.1) if len(ult)>=13 else 1.0,
                    0.05, 20.0)),
                'damage_enc':    lm('damage_enc'),
                'repair_enc':    lm('repair_enc'),
                'country_enc':   lm('country_enc'),
                'avg_age_days':  lm('avg_age_days') + 30,
                'avg_engine_age':lm('avg_engine_age') + 30,
                'avg_km_per_mis':lm('avg_km_per_mis'),
                'avg_displacement':lm('avg_displacement'),
                'avg_repeat_rate': lm('avg_repeat_rate'),
            }
            row['mom'] = row['lag1'] - row['lag2']
            filas_pred.append(row)

        if not filas_pred: break
        df_p = pd.DataFrame(filas_pred)
        pc   = np.maximum(0, mc.predict(df_p[FEATS]))
        pl   = np.maximum(0, ml.predict(df_p[FEATS]))
        ph   = np.maximum(0, mu.predict(df_p[FEATS]))
        df_p['fallas_forecast'] = np.round(pc * df_p['scale']).astype(int)
        df_p['lower_89']        = np.round(pl * df_p['scale']).astype(int)
        df_p['upper_89']        = np.round(ph * df_p['scale']).astype(int)
        df_p['paso']            = paso
        df_p['fallas_w']        = pc * df_p['scale']
        df_p['global_idx']      = df_p['g_idx_lag1'] + 0.02
        df_p['fallas']          = df_p['fallas_forecast']
        forecasts.append(df_p[['mes','cod_enc','tma_enc','engine_enc','mis_enc',
                                'paso','fallas_forecast','lower_89','upper_89']])
        cols_int = list(set(historia.columns) & set(df_p.columns))
        historia = pd.concat([historia, df_p[cols_int]], ignore_index=True)
        print(f'   Paso +{paso}: {next_month.strftime("%B %Y")} → {len(df_p):,} combinaciones | Total: {df_p["fallas_forecast"].sum():,} fallas')

    fc = pd.concat(forecasts, ignore_index=True)
    fc['codigo']      = le_cod.inverse_transform(fc['cod_enc'].astype(int))
    fc['tma']         = le_tma.inverse_transform(fc['tma_enc'].astype(int))
    fc['mis_cluster'] = le_mis.inverse_transform(fc['mis_enc'].astype(int))
    return fc

forecast_3m = forecast_recursivo(monthly, m_central, m_lower, m_upper, N_MESES_FC)
print(f'\n✅ Forecast generado: {len(forecast_3m):,} filas')

## Celda 7 — Costos, Top 50 y Alertas

In [ ]:
def ponderar_con_costos(fc, ref, fallback):
    fc = fc.merge(ref[['codigo','tma','costo_unit_med','costo_unit_avg',
                        'costo_unit_p90','material_unit','labour_unit']],
                  on=['codigo','tma'], how='left')
    med_global = pd.Series(fallback).median()
    fc['costo_unit_med'] = fc.apply(
        lambda r: r['costo_unit_med'] if pd.notna(r['costo_unit_med'])
        else fallback.get(r['codigo'], med_global), axis=1)
    fc['costo_unit_avg'] = fc['costo_unit_avg'].fillna(fc['costo_unit_med'])
    fc['costo_unit_p90'] = fc['costo_unit_p90'].fillna(fc['costo_unit_med']*1.5)
    fc['material_unit']  = fc['material_unit'].fillna(fc['costo_unit_med']*0.22)
    fc['labour_unit']    = fc['labour_unit'].fillna(fc['costo_unit_med']*0.68)
    fc['costo_proyectado']    = fc['fallas_forecast'] * fc['costo_unit_med']
    fc['costo_lower']         = fc['lower_89']        * fc['costo_unit_med']
    fc['costo_upper']         = fc['upper_89']        * fc['costo_unit_med']
    fc['costo_pesimista']     = fc['upper_89']        * fc['costo_unit_p90']
    fc['material_proyectado'] = fc['fallas_forecast'] * fc['material_unit']
    fc['labour_proyectado']   = fc['fallas_forecast'] * fc['labour_unit']
    return fc

forecast_3m = ponderar_con_costos(forecast_3m, costos_ref, costos_fallback)

def calcular_top_n(fc, n=50):
    top = fc.groupby(['codigo','tma','mis_cluster']).agg(
        fallas_3m=('fallas_forecast','sum'), upper_89_3m=('upper_89','sum'),
        lower_89_3m=('lower_89','sum'), costo_proyectado_3m=('costo_proyectado','sum'),
        costo_pesimista_3m=('costo_pesimista','sum'), material_3m=('material_proyectado','sum'),
        labour_3m=('labour_proyectado','sum'), costo_unit_med=('costo_unit_med','first'),
    ).reset_index()
    for val, sfx in [('fallas_forecast',['fallas_M1','fallas_M2','fallas_M3']),
                     ('costo_proyectado',['costo_M1','costo_M2','costo_M3'])]:
        pv = fc.pivot_table(index=['codigo','tma','mis_cluster'], columns='paso',
                             values=val, aggfunc='sum').reset_index()
        pv.columns = ['codigo','tma','mis_cluster'] + sfx
        top = top.merge(pv, on=['codigo','tma','mis_cluster'])
    top = top.sort_values('costo_proyectado_3m', ascending=False).head(n).reset_index(drop=True)
    top['ranking'] = range(1, len(top)+1)
    top['pct_costo_total'] = (top['costo_proyectado_3m']/top['costo_proyectado_3m'].sum()*100).round(2)
    return top

top50 = calcular_top_n(forecast_3m, TOP_N)

def generar_alertas(monthly, fc):
    umbrales = dict(
        fc_vs_hist_alto=2.0, fc_vs_hist_medio=1.5,
        tendencia_alto=2.0,  tendencia_medio=1.4,
        costo_critico=500_000, costo_alto=200_000, costo_medio=100_000,
        mis_temprano_costo=50_000,
        concentracion_alta=5, concentracion_media=3,
        zscore_alto=3, zscore_medio=2,
        costo_unit_alto=3000,
    )
    hist = monthly.groupby(['codigo','tma','mis_cluster']).agg(
        media_hist=('fallas','mean'), std_hist=('fallas','std'),
        max_hist=('fallas','max'),   meses_data=('fallas','count')
    ).reset_index()
    cf = monthly['mes'].max() - pd.DateOffset(months=3)
    rec = monthly[monthly['mes']>cf].groupby(['codigo','tma','mis_cluster'])['fallas'].mean().reset_index().rename(columns={'fallas':'media_reciente'})
    pre = monthly[monthly['mes']<=cf].groupby(['codigo','tma','mis_cluster'])['fallas'].mean().reset_index().rename(columns={'fallas':'media_previo'})
    tend = rec.merge(pre, on=['codigo','tma','mis_cluster'], how='left')
    tend['ratio_tendencia'] = tend['media_reciente'] / tend['media_previo'].replace(0,1).clip(lower=1)
    fc_agg = fc.groupby(['codigo','tma','mis_cluster']).agg(
        fallas_fc_3m=('fallas_forecast','sum'), costo_fc_3m=('costo_proyectado','sum'),
        costo_unit=('costo_unit_med','first')
    ).reset_index()
    A = (fc_agg.merge(hist, on=['codigo','tma','mis_cluster'], how='left')
               .merge(tend[['codigo','tma','mis_cluster','ratio_tendencia']], on=['codigo','tma','mis_cluster'], how='left'))
    A['ratio_tendencia'] = A['ratio_tendencia'].fillna(1.0)
    A['fc_vs_hist']    = A['fallas_fc_3m'] / (A['media_hist']*3 + 1)
    A['z_score_fc']    = (A['fallas_fc_3m']/3 - A['media_hist']) / (A['std_hist'].fillna(1)+1)
    A['concentracion'] = A['costo_fc_3m'] / A['costo_fc_3m'].sum() * 100
    A['mis_temprano']  = A['mis_cluster'] == '0-3 MIS'
    def riesgo(r):
        s, razones = 0, []
        if r['fc_vs_hist']>umbrales['fc_vs_hist_alto']:       s+=30; razones.append(f'Forecast {r["fc_vs_hist"]:.1f}× sobre histórico')
        elif r['fc_vs_hist']>umbrales['fc_vs_hist_medio']:    s+=15; razones.append(f'Forecast {r["fc_vs_hist"]:.1f}× sobre histórico')
        if r['ratio_tendencia']>umbrales['tendencia_alto']:   s+=25; razones.append(f'Aceleración reciente {r["ratio_tendencia"]:.1f}×')
        elif r['ratio_tendencia']>umbrales['tendencia_medio']:s+=12; razones.append(f'Tendencia ascendente {r["ratio_tendencia"]:.1f}×')
        if r['costo_fc_3m']>umbrales['costo_critico']:        s+=30; razones.append(f'Costo proyectado >${umbrales["costo_critico"]/1000:.0f}k')
        elif r['costo_fc_3m']>umbrales['costo_alto']:         s+=15; razones.append(f'Costo proyectado >${umbrales["costo_alto"]/1000:.0f}k')
        elif r['costo_fc_3m']>umbrales['costo_medio']:        s+=8
        if r['mis_temprano'] and r['costo_fc_3m']>umbrales['mis_temprano_costo']: s+=20; razones.append('MIS 0-3 con costo alto: posible problema de manufactura')
        if r['concentracion']>umbrales['concentracion_alta']:   s+=15; razones.append(f'Concentra {r["concentracion"]:.1f}% del costo total')
        elif r['concentracion']>umbrales['concentracion_media']:s+=8
        if r['z_score_fc']>umbrales['zscore_alto']:    s+=20; razones.append(f'Anomalía estadística (z={r["z_score_fc"]:.1f})')
        elif r['z_score_fc']>umbrales['zscore_medio']: s+=10
        if r['costo_unit']>umbrales['costo_unit_alto']: s+=10; razones.append(f'Costo unitario alto (${r["costo_unit"]:.0f})')
        nivel = '🔴 CRÍTICO' if s>=70 else ('🟠 ALTO' if s>=45 else ('🟡 MEDIO' if s>=25 else '🟢 BAJO'))
        return pd.Series({'score_riesgo':s,'nivel_alerta':nivel,'razones':' | '.join(razones) if razones else 'Sin señales'})
    A[['score_riesgo','nivel_alerta','razones']] = A.apply(riesgo, axis=1)
    return A.sort_values('score_riesgo', ascending=False).reset_index(drop=True)

alertas = generar_alertas(monthly, forecast_3m)

resumen = forecast_3m.groupby('paso').agg(
    n_comb=('fallas_forecast','count'), fallas=('fallas_forecast','sum'),
    costo_central=('costo_proyectado','sum'), costo_lower=('costo_lower','sum'),
    costo_upper=('costo_upper','sum'), material=('material_proyectado','sum'),
    labour=('labour_proyectado','sum'),
).round(0)
print('💰 COSTOS PROYECTADOS A 3 MESES (Material + Labour)')
print(resumen.to_string())
print(f'\n   TOTAL Material: $ {resumen["material"].sum():>15,.0f}')
print(f'   TOTAL Labour:   $ {resumen["labour"].sum():>15,.0f}')
print(f'   TOTAL Central:  $ {resumen["costo_central"].sum():>15,.0f}')
print('\n🚨 ALERTAS:')
for nivel in ['🔴 CRÍTICO','🟠 ALTO','🟡 MEDIO','🟢 BAJO']:
    n = (alertas['nivel_alerta']==nivel).sum()
    c = alertas[alertas['nivel_alerta']==nivel]['costo_fc_3m'].sum()
    print(f'   {nivel}: {n:4d} | ${c:>12,.0f}')

## Celda 8 — Exportar CSVs para Power BI + guardar modelo

In [ ]:
# ── Tabla unificada histórico + forecast ──────────────────────────────────
hist_exp = monthly[['mes','codigo','tma','mis_cluster','fallas','costo_total',
                     'material_total','labour_total']].copy()
hist_exp['tipo']            = 'Historico'
hist_exp.rename(columns={'fallas':'fallas_reales'}, inplace=True)
hist_exp['fallas_forecast'] = np.nan
hist_exp['lower_89']        = np.nan
hist_exp['upper_89']        = np.nan
hist_exp['costo_proyectado']= np.nan
hist_exp['paso']            = 0

fc_exp = forecast_3m[['mes','codigo','tma','mis_cluster','paso',
                       'fallas_forecast','lower_89','upper_89',
                       'costo_proyectado','material_proyectado','labour_proyectado']].copy()
fc_exp['tipo']          = 'Forecast'
fc_exp['fallas_reales'] = np.nan
fc_exp['costo_total']   = np.nan
fc_exp.rename(columns={'material_proyectado':'material_total',
                        'labour_proyectado':'labour_total'}, inplace=True)

cols_u = ['mes','codigo','tma','mis_cluster','tipo','paso','fallas_reales',
          'fallas_forecast','lower_89','upper_89','costo_total',
          'costo_proyectado','material_total','labour_total']
tabla_unificada = pd.concat([hist_exp[cols_u], fc_exp[cols_u]], ignore_index=True)
tabla_unificada['mes'] = pd.to_datetime(tabla_unificada['mes'])
tabla_unificada = tabla_unificada.sort_values(['codigo','tma','mis_cluster','mes']).reset_index(drop=True)

tabla_unificada.to_csv(OUTPUT_DIR/'forecast_completo.csv', index=False, encoding='utf-8-sig')
top50.to_csv(OUTPUT_DIR/'top50_costos.csv', index=False, encoding='utf-8-sig')
alertas.to_csv(OUTPUT_DIR/'alertas_tempranas.csv', index=False, encoding='utf-8-sig')

# ── Versionar modelo previo y guardar nuevo ───────────────────────────────
if hay_modelo_previo:
    meta_old = json.load(open(MODEL_DIR/'metadata.json'))
    ver_old  = meta_old.get('version','v1.0')
    arch_dir = HISTORY_DIR / ver_old
    arch_dir.mkdir(exist_ok=True)
    for f in MODEL_DIR.iterdir():
        if f.is_file(): shutil.copy2(f, arch_dir/f.name)
    print(f'📦 Modelo anterior ({ver_old}) archivado en {arch_dir}/')
    parts = ver_old[1:].split('.')
    nueva_version = f'v{parts[0]}.{int(parts[1])+1}'
else:
    nueva_version = 'v1.0'

joblib.dump(m_central, MODEL_DIR/'model_central.pkl')
joblib.dump(m_lower,   MODEL_DIR/'model_lower.pkl')
joblib.dump(m_upper,   MODEL_DIR/'model_upper.pkl')
joblib.dump({
    'le_cod':le_cod,'le_tma':le_tma,'le_mis':le_mis,
    'le_engine':le_engine,'le_damage':le_damage,
    'le_repair':le_repair,'le_country':le_country,
    'component_repeat_rate_map':crr_map,
    'global_repeat_rate':global_rr, 'feats':FEATS,
}, MODEL_DIR/'encoders.pkl')

metadata = {
    'version':nueva_version, 'modo':modo,
    'fecha':datetime.now().isoformat(),
    'accuracy':float(accuracy), 'mape':float(mape),
    'horizonte_meses':N_MESES_FC, 'n_features':len(FEATS),
    'n_combinaciones':int(forecast_3m.groupby(['codigo','tma','mis_cluster']).ngroups),
    'costo_material_proyectado':float(forecast_3m['material_proyectado'].sum()),
    'costo_labour_proyectado':float(forecast_3m['labour_proyectado'].sum()),
    'costo_total_proyectado':float(forecast_3m['costo_proyectado'].sum()),
    'alertas_criticas':int((alertas['nivel_alerta']=='🔴 CRÍTICO').sum()),
    'alertas_altas':int((alertas['nivel_alerta']=='🟠 ALTO').sum()),
}
with open(MODEL_DIR/'metadata.json','w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f'💾 Modelo {nueva_version} guardado')
print(f'✅ CSVs Power BI exportados en {OUTPUT_DIR}/')
print(f'   • forecast_completo.csv  ({len(tabla_unificada):,} filas)')
print(f'   • top50_costos.csv       ({len(top50)} filas)')
print(f'   • alertas_tempranas.csv  ({len(alertas)} filas)')

## Celda 9 — VIZ 1: Línea de tiempo total con bandas de confianza

In [ ]:
def viz1_timeline_total(monthly, forecast_3m, output_dir):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14,9),
                                    gridspec_kw={'height_ratios':[1,1],'hspace':0.45})
    hist = monthly.groupby('mes').agg(fallas=('fallas','sum')).reset_index()
    fc   = forecast_3m.groupby('mes').agg(
        fallas=('fallas_forecast','sum'), upper=('upper_89','sum'), lower=('lower_89','sum')
    ).reset_index()
    conectar = pd.concat([hist.tail(1), fc.rename(columns={'fallas':'fallas'}).head(1)])
    ax1.fill_between(fc['mes'], fc['lower'], fc['upper'], color=COLOR_GOLD, alpha=0.20, label='Banda 89%')
    ax1.plot(hist['mes'], hist['fallas'], color=COLOR_NAVY, linewidth=2.5, label='Histórico real')
    ax1.plot(conectar['mes'], conectar['fallas'], color=COLOR_GOLD, linewidth=2.5, linestyle='--')
    ax1.plot(fc['mes'], fc['fallas'], color=COLOR_GOLD, linewidth=2.5, linestyle='--',
             marker='D', markersize=10, label='Forecast 3 meses')
    trans = hist['mes'].max()
    ax1.axvline(x=trans, color=COLOR_GRAY_M, linestyle=':', linewidth=1.5, alpha=0.7)
    ax1.text(trans, ax1.get_ylim()[1]*0.95, '  Forecast →', fontsize=10,
             color=COLOR_GRAY_M, fontstyle='italic')
    ax1.set_ylabel('Fallas mensuales', fontweight='bold')
    ax1.legend(loc='upper left')
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
    n_meses_hist = monthly['mes'].dt.to_period('M').nunique()
    styled_title(ax1,'Trayectoria de fallas en campo',
                 f'Histórico de {n_meses_hist} meses + forecast recursivo a 3 meses con banda 89%')

    hist_c = monthly.groupby('mes').agg(costo=('costo_total','sum')).reset_index()
    fc_c   = forecast_3m.groupby('mes').agg(
        costo=('costo_proyectado','sum'), upper=('costo_upper','sum'), lower=('costo_lower','sum')
    ).reset_index()
    conectar_c = pd.concat([hist_c.tail(1), fc_c.head(1)])
    ax2.fill_between(fc_c['mes'], fc_c['lower'], fc_c['upper'], color=COLOR_GOLD, alpha=0.20, label='Banda 89%')
    ax2.plot(hist_c['mes'], hist_c['costo'], color=COLOR_NAVY, linewidth=2.5, label='Costo histórico')
    ax2.plot(conectar_c['mes'], conectar_c['costo'], color=COLOR_GOLD, linewidth=2.5, linestyle='--')
    ax2.plot(fc_c['mes'], fc_c['costo'], color=COLOR_GOLD, linewidth=2.5, linestyle='--',
             marker='D', markersize=10, label='Costo proyectado')
    ax2.axvline(x=trans, color=COLOR_GRAY_M, linestyle=':', linewidth=1.5, alpha=0.7)
    total_3m = fc_c['costo'].sum()
    ax2.annotate(f'$ {total_3m/1e6:.1f}M\nproyectados\n3 meses',
                 xy=(fc_c['mes'].iloc[-1], fc_c['costo'].iloc[-1]),
                 xytext=(15,30), textcoords='offset points',
                 fontsize=11, fontweight='bold', color=COLOR_NAVY,
                 bbox=dict(boxstyle='round,pad=0.6', fc=COLOR_LIGHT, ec=COLOR_GOLD, lw=1.5),
                 arrowprops=dict(arrowstyle='->', color=COLOR_NAVY, lw=1.2))
    ax2.set_ylabel('Costo USD', fontweight='bold'); ax2.set_xlabel('Mes')
    ax2.legend(loc='upper left')
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_currency_k))
    styled_title(ax2,'Trayectoria de costo total de fallas',
                 f'Costo histórico: ${monthly["costo_total"].sum()/1e6:.1f}M  |  Proyección 3M: ${total_3m/1e6:.1f}M')
    plt.savefig(output_dir/'viz1_timeline_total.png'); plt.show(); return fig

fig1 = viz1_timeline_total(monthly, forecast_3m, OUTPUT_DIR)

## Celda 10 — VIZ 2: Concentración por TMA y MIS Cluster

In [ ]:
def viz2_concentracion(monthly, forecast_3m, output_dir):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15,6))
    hist_tma = monthly.groupby('tma')['costo_total'].sum().sort_values()
    fc_tma   = forecast_3m.groupby('tma')['costo_proyectado'].sum().reindex(hist_tma.index)
    y = np.arange(len(hist_tma)); h = 0.4
    ax1.barh(y-h/2, hist_tma.values, height=h, color=COLOR_NAVY, label='Histórico')
    ax1.barh(y+h/2, fc_tma.values,   height=h, color=COLOR_GOLD, label='Forecast 3M')
    for i,(hv,fv) in enumerate(zip(hist_tma.values, fc_tma.values)):
        ax1.text(hv, i-h/2, f'  ${hv/1e6:.2f}M', va='center', fontsize=9, color=COLOR_GRAY_D)
        ax1.text(fv, i+h/2, f'  ${fv/1e6:.2f}M', va='center', fontsize=9, color=COLOR_GRAY_D)
    ax1.set_yticks(y); ax1.set_yticklabels(hist_tma.index, fontweight='bold')
    ax1.set_xlabel('Costo total USD'); ax1.legend(loc='lower right')
    ax1.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_currency_k))
    ax1.grid(axis='y', visible=False)
    styled_title(ax1,'Costo por modelo de vehículo (TMA)','Histórico vs proyectado — identifica concentración')

    mis_order = ['0-3 MIS','4-12 MIS','13-24 MIS']
    hist_mis = monthly.groupby('mis_cluster')['costo_total'].sum().reindex(mis_order)
    fc_mis   = forecast_3m.groupby('mis_cluster')['costo_proyectado'].sum().reindex(mis_order)
    x = np.arange(3); bw = 0.35
    ax2.bar(x-bw/2, hist_mis.values, width=bw, color=COLOR_NAVY, label='Histórico')
    ax2.bar(x+bw/2, fc_mis.values,   width=bw, color=COLOR_GOLD, label='Forecast 3M')
    th = hist_mis.sum(); tf = fc_mis.sum()
    for i,(hv,fv) in enumerate(zip(hist_mis.values, fc_mis.values)):
        ax2.text(i-bw/2, hv, f'{hv/th*100:.0f}%', ha='center', va='bottom', fontweight='bold', fontsize=10, color=COLOR_NAVY)
        ax2.text(i+bw/2, fv, f'{fv/tf*100:.0f}%', ha='center', va='bottom', fontweight='bold', fontsize=10, color=COLOR_GOLD)
    yl = ax2.get_ylim()[1]
    ax2.text(0,yl*0.85,'Manufactura',   ha='center', fontsize=9, color=COLOR_RED,    fontweight='bold')
    ax2.text(1,yl*0.85,'Diseño /\nproveedores',ha='center', fontsize=9, color=COLOR_ORANGE, fontweight='bold')
    ax2.text(2,yl*0.85,'Desgaste\nprematuro',  ha='center', fontsize=9, color=COLOR_TEAL,   fontweight='bold')
    ax2.set_xticks(x); ax2.set_xticklabels(mis_order, fontweight='bold')
    ax2.set_ylabel('Costo USD'); ax2.legend(loc='upper left')
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_currency_k))
    ax2.grid(axis='x', visible=False)
    styled_title(ax2,'Costo por madurez del motor (MIS)','Cada cluster sugiere un tipo distinto de causa raíz')
    plt.tight_layout(); plt.savefig(output_dir/'viz2_concentracion_tma_mis.png'); plt.show(); return fig

fig2 = viz2_concentracion(monthly, forecast_3m, OUTPUT_DIR)

## Celda 11 — VIZ 3: Pareto del Top 50

In [ ]:
def viz3_pareto_top50(top50, output_dir):
    fig, ax1 = plt.subplots(figsize=(14,7))
    n = len(top50); x = np.arange(n)
    colors = [COLOR_RED if i<10 else (COLOR_ORANGE if i<20 else (COLOR_GOLD if i<30 else COLOR_TEAL)) for i in range(n)]
    ax1.bar(x, top50['costo_proyectado_3m'], color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax1.set_ylabel('Costo proyectado 3 meses (USD)', fontweight='bold')
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_currency_k))
    ax2 = ax1.twinx()
    cum_pct = top50['pct_costo_total'].cumsum()
    ax2.plot(x, cum_pct, color=COLOR_NAVY, linewidth=2.5, marker='o', markersize=4, label='% acumulado')
    ax2.set_ylabel('% acumulado del costo Top 50', fontweight='bold', color=COLOR_NAVY)
    ax2.set_ylim(0,105); ax2.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
    ax2.tick_params(axis='y', labelcolor=COLOR_NAVY); ax2.grid(False); ax2.spines['right'].set_visible(True)
    ax2.axhline(y=80, color=COLOR_GRAY_M, linestyle=':', linewidth=1.2)
    ax2.text(n*0.95, 81, '80%', fontsize=10, color=COLOR_GRAY_M, fontweight='bold', ha='right')
    idx_80 = (cum_pct < 80).sum()
    if idx_80 < n:
        ax1.axvline(x=idx_80-0.5, color=COLOR_RED, linestyle='--', linewidth=1.5, alpha=0.6)
        ax1.text(idx_80-0.3, ax1.get_ylim()[1]*0.95,
                 f'  El Top {idx_80} concentra el 80%\n  del costo proyectado', fontsize=10, color=COLOR_RED, fontweight='bold')
    ax1.set_xticks(x[::5]); ax1.set_xticklabels([f'#{i+1}' for i in x[::5]], fontsize=9)
    ax1.set_xlabel('Ranking (Top 50 por costo proyectado 3M)')
    legend_patches = [
        mpatches.Patch(color=COLOR_RED, label='Top 10'),
        mpatches.Patch(color=COLOR_ORANGE, label='Top 11-20'),
        mpatches.Patch(color=COLOR_GOLD, label='Top 21-30'),
        mpatches.Patch(color=COLOR_TEAL, label='Top 31-50'),
    ]
    ax1.legend(handles=legend_patches, loc='upper right', ncol=2, framealpha=0.95)
    styled_title(ax1,'Análisis de Pareto del Top 50',
                 f'Top 10 = {top50.head(10)["pct_costo_total"].sum():.0f}%  |  Top 20 = {top50.head(20)["pct_costo_total"].sum():.0f}%')
    plt.tight_layout(); plt.savefig(output_dir/'viz3_pareto_top50.png'); plt.show(); return fig

fig3 = viz3_pareto_top50(top50, OUTPUT_DIR)

## Celda 12 — VIZ 4: Heatmap código × TMA

In [ ]:
def viz4_heatmap_codigo_tma(forecast_3m, top_n=20, output_dir=None):
    top_cod = forecast_3m.groupby('codigo')['costo_proyectado'].sum().sort_values(ascending=False).head(top_n).index.tolist()
    pivot = (forecast_3m[forecast_3m['codigo'].isin(top_cod)]
             .pivot_table(index='codigo', columns='tma', values='costo_proyectado', aggfunc='sum')
             .fillna(0).reindex(top_cod))
    fig, ax = plt.subplots(figsize=(13,9))
    cmap = LinearSegmentedColormap.from_list('silao',
        ['#FFFFFF','#F7F5F0','#F39C12','#E67E22','#C0392B'], N=256)
    sns.heatmap(pivot/1000, annot=True, fmt='.0f', cmap=cmap, ax=ax,
                cbar_kws={'label':'Costo proyectado 3M (miles USD)','shrink':0.7},
                linewidths=0.5, linecolor='white', annot_kws={'fontsize':9,'fontweight':'bold'})
    ax.set_xlabel('Modelo de vehículo (TMA)', fontweight='bold')
    ax.set_ylabel('Código de falla', fontweight='bold')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    plt.setp(ax.yaxis.get_majorticklabels(), rotation=0)
    styled_title(ax, f'Mapa de calor: Top {top_n} códigos × TMA',
                 'Hot spots — celdas más oscuras = mayor costo proyectado')
    plt.tight_layout()
    if output_dir: plt.savefig(output_dir/'viz4_heatmap_codigo_tma.png')
    plt.show(); return fig

fig4 = viz4_heatmap_codigo_tma(forecast_3m, top_n=20, output_dir=OUTPUT_DIR)

## Celda 13 — VIZ 5: Anatomía del costo (material vs labour) en Top 15

In [ ]:
def viz5_material_vs_labour(top50, output_dir):
    top15 = top50.head(15).copy().iloc[::-1]
    top15['etiqueta'] = top15.apply(lambda r: f"{r['codigo'][:25]}\n{r['tma']} ({r['mis_cluster']})", axis=1)
    fig, ax = plt.subplots(figsize=(14,8))
    y = np.arange(len(top15))
    otros = (top15['costo_proyectado_3m'] - top15['material_3m'] - top15['labour_3m']).clip(lower=0)
    ax.barh(y, top15['material_3m'], color=COLOR_TEAL, label='Material', edgecolor='white', linewidth=1)
    ax.barh(y, top15['labour_3m'], left=top15['material_3m'], color=COLOR_GOLD, label='Mano de obra', edgecolor='white', linewidth=1)
    ax.barh(y, otros, left=top15['material_3m']+top15['labour_3m'], color=COLOR_GRAY_M, label='Overhead/garantía', edgecolor='white', linewidth=1)
    for i, (mat,lab,otr,tot) in enumerate(zip(top15['material_3m'],top15['labour_3m'],otros,top15['costo_proyectado_3m'])):
        ax.text(tot, i, f'  ${tot/1000:.0f}k', va='center', fontsize=9, color=COLOR_NAVY, fontweight='bold')
    ax.set_yticks(y); ax.set_yticklabels(top15['etiqueta'], fontsize=9)
    ax.set_xlabel('Costo proyectado 3 meses (USD)', fontweight='bold')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_currency_k))
    ax.legend(loc='lower right', ncol=3); ax.grid(axis='y', visible=False)
    styled_title(ax,'Top 15 combinaciones — Anatomía del costo',
                 'Desglose material, mano de obra y overhead — identifica si el problema es de pieza o trabajo')
    plt.tight_layout(); plt.savefig(output_dir/'viz5_anatomia_costo.png'); plt.show(); return fig

fig5 = viz5_material_vs_labour(top50, OUTPUT_DIR)

## Celda 14 — VIZ 6: Matriz de riesgo (impacto × aceleración)

In [ ]:
def viz6_matriz_riesgo(alertas, output_dir):
    df = alertas.copy()
    df['acceleracion'] = df['ratio_tendencia'].clip(0,8)
    df['impacto_log']  = np.log10(df['costo_fc_3m'].clip(lower=100))
    fig, ax = plt.subplots(figsize=(13,9))
    ymax = df['impacto_log'].max()*1.05
    ax.axhspan(np.log10(200_000), ymax, xmin=0.5, xmax=1, color=COLOR_RED,    alpha=0.06, zorder=0)
    ax.axhspan(np.log10(50_000), np.log10(200_000), xmin=0.5, xmax=1, color=COLOR_ORANGE, alpha=0.06, zorder=0)
    ax.axhspan(np.log10(200_000), ymax, xmin=0,   xmax=0.5, color=COLOR_YELLOW, alpha=0.06, zorder=0)
    for nivel, color in MAPA_NIVELES.items():
        mask = df['nivel_alerta'] == nivel
        sizes = (df.loc[mask,'fallas_fc_3m']/15).clip(20,600)
        ax.scatter(df.loc[mask,'acceleracion'], df.loc[mask,'impacto_log'],
                   s=sizes, color=color, alpha=0.6, edgecolors='white',
                   linewidth=1, label=MAPA_NIVELES_TEXTO[nivel], zorder=3)
    for _, r in df[df['nivel_alerta']=='🔴 CRÍTICO'].head(8).iterrows():
        ax.annotate(f"{r['codigo'][:18]}\n{r['tma'][:10]}",
                    xy=(min(r['ratio_tendencia'],8), np.log10(max(r['costo_fc_3m'],100))),
                    xytext=(8,5), textcoords='offset points', fontsize=8, color=COLOR_GRAY_D,
                    bbox=dict(boxstyle='round,pad=0.3',fc='white',ec=COLOR_RED,lw=0.7,alpha=0.85))
    ax.axvline(x=2, color=COLOR_GRAY_M, linestyle=':', linewidth=1, alpha=0.7)
    ax.axhline(y=np.log10(200_000), color=COLOR_GRAY_M, linestyle=':', linewidth=1, alpha=0.7)
    for txt, xa, ya, c in [
        ('ACCIÓN\nINMEDIATA',0.97,0.97,COLOR_RED),('MONITOREAR',0.03,0.97,COLOR_YELLOW),
        ('PRIORIZAR',0.97,0.03,COLOR_ORANGE),('BAJA\nPRIORIDAD',0.03,0.03,COLOR_GREEN)
    ]:
        ha = 'right' if xa>0.5 else 'left'; va = 'top' if ya>0.5 else 'bottom'
        ax.text(xa, ya, txt, transform=ax.transAxes, fontsize=12, fontweight='bold',
                color=c, ha=ha, va=va, bbox=dict(boxstyle='round,pad=0.4',fc='white',ec=c,lw=1))
    ax.set_xlabel('Aceleración de tendencia (últimos 3M / anteriores)', fontweight='bold')
    ax.set_ylabel('Impacto económico — costo proyectado 3M (escala log)', fontweight='bold')
    ticks_r = [1_000,10_000,100_000,500_000,1_000_000]
    ax.set_yticks([np.log10(v) for v in ticks_r])
    ax.set_yticklabels([f'${v/1000:.0f}k' if v<1e6 else f'${v/1e6:.1f}M' for v in ticks_r])
    ax.legend(title='Nivel de alerta', loc='center right', framealpha=0.95)
    styled_title(ax,'Matriz de riesgo: aceleración × impacto económico',
                 'Tamaño = volumen de fallas | Acción urgente en cuadrante superior derecho')
    plt.tight_layout(); plt.savefig(output_dir/'viz6_matriz_riesgo.png'); plt.show(); return fig

fig6 = viz6_matriz_riesgo(alertas, OUTPUT_DIR)

## Celda 15 — VIZ 7: Top 10 alertas con razones explicadas

In [ ]:
def viz7_top_alertas_explicadas(alertas, output_dir):
    top10 = alertas[alertas['nivel_alerta'].isin(['🔴 CRÍTICO','🟠 ALTO'])].head(10)
    top10 = top10.sort_values('costo_fc_3m', ascending=True).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(15,9))
    y = np.arange(len(top10))
    colors = [MAPA_NIVELES[n] for n in top10['nivel_alerta']]
    ax.barh(y, top10['costo_fc_3m'], color=colors, alpha=0.85, edgecolor='white', linewidth=1)
    for i, r in top10.iterrows():
        ax.text(-ax.get_xlim()[1]*0.005, i+0.18, str(r['codigo']),
                ha='right', va='center', fontsize=10, fontweight='bold', color=COLOR_NAVY)
        ax.text(-ax.get_xlim()[1]*0.005, i-0.18,
                f"{r['tma']}  •  {r['mis_cluster']}  •  {int(r['fallas_fc_3m'])} fallas",
                ha='right', va='center', fontsize=8, color=COLOR_GRAY_M, fontstyle='italic')
        ax.text(r['costo_fc_3m'], i, f"  ${r['costo_fc_3m']/1000:.0f}k",
                va='center', fontsize=10, fontweight='bold', color=COLOR_NAVY)
        ax.text(r['costo_fc_3m']*1.01, i, f"  score: {int(r['score_riesgo'])}",
                va='center', fontsize=8, color=colors[i],
                bbox=dict(boxstyle='round,pad=0.25',fc='white',ec=colors[i],lw=1))
    max_x = ax.get_xlim()[1]; box_x = max_x*1.35
    for i, r in top10.iterrows():
        razones = r['razones'].split(' | ')
        txt = '\n'.join([f'• {z[:55]}{"..." if len(z)>55 else ""}' for z in razones[:4]])
        ax.text(box_x, i, txt, va='center', fontsize=8, color=COLOR_GRAY_D,
                bbox=dict(boxstyle='round,pad=0.5',fc=COLOR_LIGHT,ec=COLOR_GRAY_L,lw=0.5))
    ax.set_yticks(y); ax.set_yticklabels([]); ax.set_xlim(0, max_x)
    ax.set_xlabel('Costo proyectado 3 meses (USD)', fontweight='bold')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_currency_k))
    ax.grid(axis='y', visible=False)
    niveles_p = top10['nivel_alerta'].unique()
    ax.legend(handles=[mpatches.Patch(color=MAPA_NIVELES[n],label=MAPA_NIVELES_TEXTO[n]) for n in niveles_p],
              loc='lower right', framealpha=0.95, title='Nivel de alerta')
    plt.subplots_adjust(left=0.18, right=0.55)
    ax.text(-0.35, 1.06, 'Top 10 alertas: combinaciones que requieren acción',
            transform=ax.transAxes, fontsize=15, fontweight='bold', color=COLOR_NAVY)
    ax.text(-0.35, 1.02, 'Costo proyectado a 3M | Razones a la derecha',
            transform=ax.transAxes, fontsize=10, fontstyle='italic', color=COLOR_GRAY_M)
    plt.savefig(output_dir/'viz7_top_alertas_explicadas.png', bbox_inches='tight'); plt.show(); return fig

fig7 = viz7_top_alertas_explicadas(alertas, OUTPUT_DIR)

## Celda 16 — VIZ 8: Foco manufactura — fallas tempranas (MIS 0-3)

In [ ]:
def viz8_fallas_tempranas_manufactura(monthly, forecast_3m, output_dir):
    hist_03 = monthly[monthly['mis_cluster']=='0-3 MIS']
    fc_03   = forecast_3m[forecast_3m['mis_cluster']=='0-3 MIS']
    fig, axes = plt.subplots(1, 2, figsize=(15,7))
    ax1 = axes[0]
    serie_h = hist_03.groupby('mes').agg(fallas=('fallas','sum'),costo=('costo_total','sum')).reset_index()
    serie_f = fc_03.groupby('mes').agg(fallas=('fallas_forecast','sum'),costo=('costo_proyectado','sum')).reset_index()
    ax1.plot(serie_h['mes'], serie_h['fallas'], color=COLOR_NAVY, linewidth=2.5,
             marker='o', markersize=5, label='Histórico MIS 0-3')
    if len(serie_f)>0:
        conn = pd.concat([serie_h.tail(1)[['mes','fallas']], serie_f[['mes','fallas']]])
        ax1.plot(conn['mes'], conn['fallas'], color=COLOR_RED, linewidth=2.5,
                 linestyle='--', marker='D', markersize=8, label='Forecast MIS 0-3')
    ax1.fill_between(serie_h['mes'], 0, serie_h['fallas'], alpha=0.15, color=COLOR_NAVY)
    if len(serie_h)>=6:
        prim = serie_h.head(6)['fallas'].mean(); ult = serie_h.tail(6)['fallas'].mean()
        crec = (ult/max(prim,1)-1)*100
        ax1.text(0.97, 0.95, f'Crecimiento últimos 6M\nvs primeros 6M: {"+" if crec>=0 else ""}{crec:.0f}%',
                 transform=ax1.transAxes, fontsize=10, fontweight='bold',
                 color=COLOR_RED if crec>30 else COLOR_GRAY_D, va='top', ha='right',
                 bbox=dict(boxstyle='round,pad=0.5',fc=COLOR_LIGHT,ec=COLOR_GOLD,lw=1))
    ax1.set_ylabel('Fallas mensuales (MIS 0-3)', fontweight='bold'); ax1.set_xlabel('Mes')
    ax1.legend(loc='upper left')
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)
    styled_title(ax1,'Fallas tempranas (MIS 0-3) — señal de manufactura',
                 'Fallas en los primeros 3 meses de servicio del motor')
    ax2 = axes[1]
    top_03 = fc_03.groupby('codigo').agg(fallas=('fallas_forecast','sum'),costo=('costo_proyectado','sum')).sort_values('costo',ascending=False).head(10).iloc[::-1]
    ax2.barh(top_03.index, top_03['costo'], color=COLOR_RED, alpha=0.85, edgecolor='white', linewidth=1)
    for idx, r in top_03.iterrows():
        ax2.text(r['costo'], list(top_03.index).index(idx), f'  {int(r["fallas"])} fallas  •  ${r["costo"]/1000:.0f}k',
                 va='center', fontsize=9, color=COLOR_GRAY_D, fontweight='bold')
    ax2.set_xlabel('Costo proyectado 3M (USD)', fontweight='bold')
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_currency_k))
    ax2.grid(axis='y', visible=False)
    styled_title(ax2,'Top 10 códigos en MIS 0-3','Candidatos para investigación de causa raíz en línea')
    plt.tight_layout(); plt.savefig(output_dir/'viz8_manufactura_mis_temprano.png'); plt.show(); return fig

fig8 = viz8_fallas_tempranas_manufactura(monthly, forecast_3m, OUTPUT_DIR)

## Celda 17 — VIZ 9: KPI Cards ejecutivas

In [ ]:
def viz9_kpi_cards(monthly, forecast_3m, alertas, accuracy, output_dir):
    fig, axes = plt.subplots(2, 4, figsize=(16,7))
    fig.suptitle('Tablero ejecutivo Silao — Forecast & Alertas',
                 fontsize=16, fontweight='bold', color=COLOR_NAVY, y=1.02)
    total_costo_fc  = forecast_3m['costo_proyectado'].sum()
    n_criticas      = (alertas['nivel_alerta']=='🔴 CRÍTICO').sum()
    n_altas         = (alertas['nivel_alerta']=='🟠 ALTO').sum()
    pct_crit        = alertas[alertas['nivel_alerta']=='🔴 CRÍTICO']['costo_fc_3m'].sum() / total_costo_fc * 100
    fallas_fc       = forecast_3m['fallas_forecast'].sum()
    fallas_03       = forecast_3m[forecast_3m['mis_cluster']=='0-3 MIS']['fallas_forecast'].sum()
    pct_03          = fallas_03 / fallas_fc * 100
    n_comb          = forecast_3m.groupby(['codigo','tma','mis_cluster']).ngroups
    cards = [
        ('Accuracy del modelo',    f'{accuracy:.1f}%',         'validación interna',        COLOR_GREEN,  0),
        ('Combinaciones forecast', f'{n_comb:,}',              'código × TMA × MIS',        COLOR_TEAL,   1),
        ('Fallas proyectadas 3M',  f'{int(fallas_fc):,}',      'horizonte recursivo',       COLOR_NAVY,   2),
        ('Costo proyectado 3M',    f'${total_costo_fc/1e6:.2f}M','escenario central',       COLOR_GOLD,   3),
        ('Alertas críticas',       f'{n_criticas}',            'requieren acción inmediata',COLOR_RED,    4),
        ('Alertas altas',          f'{n_altas}',               'monitoreo cercano',         COLOR_ORANGE, 5),
        ('Costo en críticas',      f'{pct_crit:.0f}%',         'del total proyectado',      COLOR_RED,    6),
        ('Fallas tempranas MIS 0-3',f'{pct_03:.0f}%',          'foco manufactura',          COLOR_TEAL,   7),
    ]
    for titulo, valor, unidad, color, idx in cards:
        ax = axes.flat[idx]
        ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis('off')
        ax.add_patch(FancyBboxPatch((0.04,0.05), 0.92, 0.9,
            boxstyle='round,pad=0.02,rounding_size=0.04',
            fc=COLOR_LIGHT, ec=color, lw=2, transform=ax.transAxes))
        ax.add_patch(FancyBboxPatch((0.04,0.05), 0.06, 0.9,
            boxstyle='round,pad=0,rounding_size=0.04',
            fc=color, ec=color, transform=ax.transAxes))
        ax.text(0.5, 0.78, titulo, transform=ax.transAxes,
                fontsize=10, fontweight='bold', color=COLOR_GRAY_M, ha='center', va='center')
        ax.text(0.5, 0.45, valor, transform=ax.transAxes,
                fontsize=32, fontweight='bold', color=color, ha='center', va='center')
        ax.text(0.5, 0.18, unidad, transform=ax.transAxes,
                fontsize=9, fontstyle='italic', color=COLOR_GRAY_M, ha='center', va='center')
    plt.tight_layout()
    plt.savefig(output_dir/'viz9_kpi_cards.png', bbox_inches='tight'); plt.show(); return fig

fig9 = viz9_kpi_cards(monthly, forecast_3m, alertas, accuracy, OUTPUT_DIR)

## Celda 18 — Resumen final y rollback

In [ ]:
print('\n' + '═'*72)
print('  ✅ PROCESO COMPLETO V4 TERMINADO')
print('═'*72)
print(f'\n📊 RESULTADOS:')
print(f'   Modo de ejecución:              {modo}')
print(f'   Versión del modelo:             {nueva_version}')
print(f'   Features del modelo:            {len(FEATS)}')
print(f'   Accuracy holdout (fallas≥20):   {accuracy:.1f}%')
print(f'   Combinaciones forecasteadas:    {forecast_3m.groupby(["codigo","tma","mis_cluster"]).ngroups:,}')
print(f'   Fallas proyectadas 3M:          {int(forecast_3m["fallas_forecast"].sum()):,}')
print(f'   Costo material proyectado 3M:   ${forecast_3m["material_proyectado"].sum():,.0f}')
print(f'   Costo labour proyectado 3M:     ${forecast_3m["labour_proyectado"].sum():,.0f}')
print(f'   Costo total proyectado 3M:      ${forecast_3m["costo_proyectado"].sum():,.0f}')
print(f'   Alertas críticas:               {(alertas["nivel_alerta"]=="🔴 CRÍTICO").sum()}')
print(f'   Alertas altas:                  {(alertas["nivel_alerta"]=="🟠 ALTO").sum()}')
print(f'\n📁 ARCHIVOS POWER BI ({OUTPUT_DIR}):')
print(f'   ├── forecast_completo.csv    ({len(tabla_unificada):,} filas — histórico + forecast)')
print(f'   ├── top50_costos.csv         ({len(top50)} filas)')
print(f'   ├── alertas_tempranas.csv    ({len(alertas)} filas)')
for i in range(1,10):
    print(f'   ├── viz{i}_*.png')
print(f'\n📁 MODELO ({MODEL_DIR}):')
print(f'   ├── model_central/lower/upper.pkl')
print(f'   ├── encoders.pkl  (7 encoders + component_repeat_rate + FEATS)')
print(f'   └── metadata.json  (versión {nueva_version})')
if hay_modelo_previo:
    print(f'\n   Rollback disponible en {HISTORY_DIR}/')

# ── Función de rollback ────────────────────────────────────────────────────
def rollback_modelo(version_destino):
    """Restaura una versión anterior. Uso: rollback_modelo('v1.0')"""
    arch = HISTORY_DIR / version_destino
    if not arch.exists():
        vers = sorted([d.name for d in HISTORY_DIR.iterdir() if d.is_dir()])
        print(f'❌ Versión no encontrada. Disponibles: {vers}'); return False
    print(f'⏪ Restaurando a {version_destino}...')
    for f in arch.iterdir():
        if f.is_file(): shutil.copy2(f, MODEL_DIR/f.name)
    print(f'✅ Modelo restaurado a {version_destino}'); return True

# Para revertir: rollback_modelo('v1.0')
print('\n💡 Para revertir a una versión anterior: rollback_modelo("v1.0")')
print('💡 Para el próximo mes: coloca el Excel nuevo, ejecuta Run All')